In [ ]:
import ee
import ipywidgets as widgets
from uuid import uuid4


from repository_ee import (
    Exports,
    GeometryUtils,
    Landsat8,
    MapLayers,
    PeriodoAgricola,
    initialize_earth_engine,
)
PROJECT = "lateral-spirit-514212-j0"

In [ ]:
AuthenticateEarhEngine(True)


In [ ]:
# Auenticação nova
initialize_earth_engine(project=PROJECT)


In [ ]:
COORDS_TOCANTINS = [
    [-48.46995289774661, -10.859497023647087],
    [-47.84922535868411, -10.859497023647087],
    [-47.84922535868411, -10.362094203375317],
    [-48.46995289774661, -10.362094203375317],
    [-48.46995289774661, -10.859497023647087],
]
aoi = GeometryUtils.polygon(COORDS_TOCANTINS)
print(GeometryUtils.metrics(aoi))

In [ ]:
# 2) Coleção Landsat e produtos derivados.
periodo = PeriodoAgricola.tocantins(2025)
landsat = Landsat8.collection(aoi, periodo)
feature_space = Landsat8.feature_space(landsat, aoi, periodo)
pico_verde = Landsat8.green_peak(landsat, aoi, periodo)

In [ ]:
# Mapa
mapa = MapLayers.create(aoi, zoom=8)
MapLayers.add_aoi(mapa, aoi)
image_rgb = MapLayers.add_rgb(mapa, feature_space, "RGB — mediana anual")
image_false_color = MapLayers.add_false_color(mapa, feature_space, "Falsa cor — mediana anual")
#image_ndbi = MapLayers.add_ndbi(mapa, feature_space, "NDBI — mediana anual")
mapa.add_draw_control()

draw = mapa.draw_control

In [ ]:
task_rgb = Exports.image_to_drive(
    image=image_rgb,
    region=aoi,
    description="landsat_rgb"
)
task_false_color = Exports.image_to_drive(
    image=image_false_color,
    region=aoi,
    description="landsat_nir"
)
"""
task_ndbi = Exports.image_to_drive(
    image=image_ndbi,
    region=aoi,
    description="landast_ndbi"
)
"""

In [ ]:
task_rgb.start()
print("task RGB enviada para o drive")
task_rgb.status()
task_false_color.start()
print("task NIR enviada para o DRIVE")
task_false_color.status()

In [ ]:
AMARELO_PASTEL = [{
    "color": "FFEE8C",
    "fillcolor": "FFEE8C66",
}]
AZUL = [{
    "color": "4A90E2",
    "fillcolor": "4A90E266",
}]
VERDE = [{
    "color": "7ED321",
    "fillcolor": "7ED32166",
}]
CINZA = [{
    "color": "9B9B9B",
    "fillcolor": "9B9B9B66",
}]



In [ ]:
# ---------- Classe única e visualização das amostras ----------
CLASSE_NOME = "URBAN"
CLASSE_ID = 3

VIS_PARAMS_AMOSTRAS = {
    "color": CINZA[0]["color"],
    "fillColor": CINZA[0]["fillcolor"],
    "width": 2,
}
# ---------- Widgets ----------
seletor = widgets.Dropdown(
    options=[],
    description="Polígono:",
    layout=widgets.Layout(width="300px")
)

btn_excluir = widgets.Button(
    description="Excluir polígono",
    button_style="danger",
    icon="trash"
)

status = widgets.HTML("<i>Desenhe os polígonos de água.</i>")

painel = widgets.VBox([
    widgets.HTML(f"<b>Amostras: {CLASSE_NOME}</b> (ID {CLASSE_ID})"),
    seletor,
    btn_excluir,
    status,
])


In [ ]:
# ---------- Funções auxiliares ----------
def obter_geometria_selecionada():
    """Retorna a geometria cuja UUID está selecionada."""
    poligono_id = seletor.value

    if poligono_id is None:
        return None

    for geometria in draw.geometries:
        props = draw.get_geometry_properties(geometria) or {}

        if props.get("id") == poligono_id:
            return geometria

    return None


def atualizar_lista(selecionar_id=None):
    """Reconstrói a lista a partir do estado atual do DrawControl."""
    opcoes = []

    for indice, geometria in enumerate(draw.geometries, start=1):
        props = draw.get_geometry_properties(geometria) or {}

        poligono_id = props.get("id")
        classe = props.get("classe", "Sem classe")

        if poligono_id:
            opcoes.append(
                (f"{indice}. {classe}", poligono_id)
            )

    seletor.options = opcoes

    if selecionar_id and any(valor == selecionar_id for _, valor in opcoes):
        seletor.value = selecionar_id
    elif opcoes:
        seletor.value = opcoes[-1][1]


def ao_criar_poligono(_, geometry):
    """Executado automaticamente ao terminar um novo desenho."""
    poligono_id = str(uuid4())

    draw.set_geometry_properties(
        geometry,
        {
            "id": poligono_id,
            "classe": CLASSE_NOME,
            "class_id": CLASSE_ID,
        }
    )

    atualizar_lista(selecionar_id=poligono_id)
    status.value = f"<span style='color:green'>Amostra de {CLASSE_NOME} criada.</span>"


def excluir_poligono(_):
    geometria = obter_geometria_selecionada()

    if geometria is None:
        status.value = "<span style='color:red'>Selecione um polígono.</span>"
        return

    draw.remove_geometry(geometria)
    atualizar_lista()
    status.value = "<span style='color:orange'>Polígono excluído.</span>"


In [ ]:
# ---------- Eventos ----------
draw.on_geometry_create(ao_criar_poligono)
btn_excluir.on_click(excluir_poligono)


# Painel sobre o mapa
mapa.add_widget(
    painel,
    position="topright",
    opened=True,
    show_close_button=True,
    widget_icon="tag"
)

mapa

In [ ]:
amostras = mapa.user_rois.filter(ee.Filter.eq("class_id", CLASSE_ID))
amostras_quantidade = amostras.size().getInfo()
if amostras_quantidade == 0:
    raise ValueError("Desenhe pelo menos uma amostra antes de exportar.")






In [ ]:
amostras 

In [ ]:
mapa

In [ ]:
ASSET_ID_URBAN= (
    "projects/lateral-spirit-514212-j0/assets/tocantins_2025_RESERVA_URBANA"
)

task_amostras = Exports.feature_Collection_to_Assets_ee(
    collection=amostras,
    asset_id=ASSET_ID_URBAN,
    description="Amostras_URBAN_2025",
)



In [ ]:
task_amostras.start()
print(f"Exportação iniciada: {ASSET_ID_FLORESTAS}")
task_amostras.status()

In [ ]:
# As amostras são exportadas como FeatureCollection; não é necessário rasterizá-las.

In [ ]:
# Fim do fluxo. A última operação executável é a exportação do asset.